# Mental Health Dataset: Expert Annotation Tool

Welcome to the Mental Health Annotation Tool. This notebook allows you to provide clinical judgments on a pre-selected sample of statements using an interactive Likert-scale interface. Your expertise will help validate the categorical alignment of these texts while your identity remains completely anonymous through a randomized Expert ID. Thank you in advance for your time and expertise.

**Annotation:** For each of the following statements, please use the sliders to provide a score from **0.00 (Very Unlikely)** to **10.00 (Very Likely)** to indicate the categorical fit of the statement to the following categories: *Anxiety, Depression, Suicidal, Stress, Bipolar, Personality disorder, Normal*. Then push the **"Next Statement"** button.

**Upon completion:** A CSV file will automatically download to your computer. Please send that file to the study coordinator.

**To start:** Please hit the **"Run all"** (Ctrl+F9) button in the top command bar and scroll down to the Expert Annotation Interface at the bottom of this notebook.

---


In [ ]:
#@title Backend Codebase { display-mode: "form" }
import os
import random
import string
import requests
import numpy as np
import pandas as pd
import ipywidgets as widgets
from datetime import datetime
from google.colab import files
from IPython.display import display, clear_output, Javascript

def load_data():
    """
    Fetches the mental health subset CSV from a public Google Drive direct link.

    Returns:
        pd.DataFrame: The loaded dataset if successful, None otherwise.
    """
    file_id = '10UC3kDMJzP36c9eVybX683Q3DMBIDFkC'
    url = f'https://drive.google.com/uc?id={file_id}'
    try:
        return pd.read_csv(url)
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return None

def generate_expert_id():
    """
    Generates a randomized 5-character alphanumeric ID for anonymous review.

    Returns:
        str: A string consisting of 3 uppercase letters and 2 digits (e.g., 'ABC12').
    """
    letters = ''.join(random.choices(string.ascii_uppercase, k=3))
    digits = ''.join(random.choices(string.digits, k=2))
    return letters + digits

def softmax(scores):
    """
    Computes softmax probabilities from a list of numerical Likert scores.

    Args:
        scores (list or np.array): Raw 0-10 scores from the expert sliders.

    Returns:
        np.array: A probability distribution where all elements sum to 1.
    """
    scores = np.array(scores, dtype=np.float64)
    exp_scores = np.exp(scores - np.max(scores))
    return exp_scores / exp_scores.sum()

def expert_ratings(subset_df):
    """
    Initializes and manages the interactive GUI for expert statement annotation.

    Args:
        subset_df (pd.DataFrame): The dataset containing statements to be reviewed.
    """
    categories = ["Anxiety", "Normal", "Depression", "Suicidal", "Stress", "Bipolar", "Personality disorder"]
    expert_id = generate_expert_id()
    ratings = []
    index = 0

    # UI Widgets Setup
    sliders = {
        cat: widgets.FloatSlider(
            value=0.0,
            min=0.0,
            max=10.0,
            step=0.01,
            readout_format='.2f',
            description=cat,
            continuous_update=False,
            style={'description_width': '160px'},
            layout=widgets.Layout(width='60%')
        ) for cat in categories
    }

    statement_box = widgets.Textarea(
        layout=widgets.Layout(width='95%', height='120px'),
        disabled=True
    )

    id_footer = widgets.HTML(
        value=f"<p style='color: gray; font-size: 0.9em;'><b>Session Expert ID:</b> {expert_id}</p>"
    )

    next_button = widgets.Button(
        description="Next Statement",
        button_style='success'
    )

    # Holding box to center the button
    button_box = widgets.HBox(
        [next_button],
        layout=widgets.Layout(
            width='60%',
            display='flex',
            justify_content='center',
            margin='10px 0px 0px 0px'
        )
    )

    def reset_sliders():
        """Resets all slider values to zero for the next observation."""
        for s in sliders.values():
            s.value = 0

    def show_statement(i):
        """Displays current progress, the statement, and rating sliders."""
        clear_output(wait=True)
        print(f"--- Statement {i+1} of {len(subset_df)} ---")
        statement_box.value = subset_df.iloc[i]['statement']
        display(statement_box)
        display(id_footer)
        print("\nRate the categorical fit (0.00 -> Very Unlikely, 10.00 -> Very Likely):")

        # Randomize slider order for display
        sliders_list = list(sliders.values())
        random.shuffle(sliders_list)
        for s in sliders_list:
            display(s)
        display(button_box)

    def on_next_clicked(b):
        """Records slider data and moves to the next statement or concludes session."""
        nonlocal index
        current_row = {cat: sliders[cat].value for cat in categories}
        current_row.update({
            "statement_id": subset_df.iloc[index]["id"],
            "statement": subset_df.iloc[index]["statement"],
            "user_status": subset_df.iloc[index]["status"],
            "word_count": subset_df.iloc[index]["word_count"],
            "text_size": subset_df.iloc[index]["text_size"],
            "expert_id": expert_id
        })
        ratings.append(current_row)
        index += 1
        if index < len(subset_df):
            reset_sliders()
            show_statement(index)
        else:
            end_session()

    def end_session():
        """Generates results, attempts auto-download, and provides manual fallback."""
        clear_output()
        print("Processing your ratings... please wait.")
        expert_df = pd.DataFrame(ratings)

        # Statistical processing
        softmax_probs = expert_df[categories].apply(softmax, axis=1, result_type='expand')
        softmax_probs.columns = [f"p_{cat}" for cat in categories]
        expert_df = pd.concat([expert_df, softmax_probs], axis=1)
        expert_df["expert_status"] = softmax_probs.idxmax(axis=1).str.replace("p_", "")

        # File generation
        filename = f"ratings_{expert_id}_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
        expert_df.to_csv(filename, index=False)

        print("✅ Annotation Complete!")
        print("-" * 50)

        # 1. Attempt automatic download first
        try:
            files.download(filename)
        except:
            pass

        # 2. Provide manual button as fallback
        print(f"Your results file has been generated: {filename}")
        print("\nIf the file did not download automatically (it may be blocked by your browser),")
        print("please click the button below:")

        dl_button = widgets.Button(
            description="Download Ratings CSV",
            button_style='info',
            icon='download',
            layout=widgets.Layout(width='300px')
        )
        def trigger_dl(b): files.download(filename)
        dl_button.on_click(trigger_dl)
        display(dl_button)

        # 3. Final submission instructions
        print("\nFinal Step:")
        print("\nOnce downloaded, please send the file to the study coordinator.")
        print("\nThank you for your time and expertise!")

    next_button.on_click(on_next_clicked)
    show_statement(index)

def main():
    """Main entry point to run the annotation tool."""
    subset_df = load_data()
    if subset_df is not None:
        expert_ratings(subset_df)

In [ ]:
#@title Expert Annotation Interface { display-mode: "form" }
if __name__ == "__main__":
    main()


---

*Code by:* **Sergio Rojas-Galeano**'

'Facultad de Ingeniería, Universidad Distrital Francisco José de Caldas

*email*: srojas@udistrital.edu.co

January, 2026

---